# EEG Analysis using AI/ML: results walkthrough

This notebook reads the aggregate-only outputs created by `scripts/run_analysis.py`. It does not load raw EEG or expose participant-level predictions. See [`PROJECT_DOCUMENTATION.md`](../PROJECT_DOCUMENTATION.md) for the complete project record.

**Question:** How do mu/beta band-power logistic regression and CSP-LDA compare with a majority reference when entire test participants are held out?


In [ ]:
from pathlib import Path
import json
import pandas as pd

candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next(p.resolve() for p in candidates if (p / "results" / "full_cohort" / "summary.csv").exists())
RESULTS = ROOT / "results" / "full_cohort"
summary = pd.read_csv(RESULTS / "summary.csv")
comparison = pd.read_csv(RESULTS / "paired_comparison.csv")
run_info = json.loads((RESULTS / "run_info.json").read_text())
audit = json.loads((RESULTS / "provenance" / "dataset_audit.json").read_text())


## Cohort and data integrity

The audit contains aggregate counts and a cohort-level fingerprint. Per-file names and participant-level data are not part of the public evidence.


In [ ]:
print(json.dumps(run_info["dataset_summary"], indent=2))
print(json.dumps(audit, indent=2))


## Aggregate model results

Mean participant accuracy is the primary summary. Its confidence interval is bootstrapped across participants. The other metrics summarize held-out trials per seed and then average across the three seeds.


In [ ]:
summary_display = summary.copy()
for column in ["mean_subject_accuracy", "subject_accuracy_ci95_low", "subject_accuracy_ci95_high", "mean_trial_accuracy", "balanced_accuracy", "macro_f1", "roc_auc"]:
    summary_display[column] = summary_display[column].map(lambda value: f"{value:.1%}")
print(summary_display.to_string(index=False))


In [ ]:
comparison_display = comparison.copy()
for column in ["mean_subject_accuracy_difference", "difference_ci95_low", "difference_ci95_high"]:
    comparison_display[column] = comparison_display[column].map(lambda value: f"{value:.1%}")
comparison_display["wilcoxon_p_value_exploratory"] = comparison_display["wilcoxon_p_value_exploratory"].map(lambda value: f"{value:.4f}")
print(comparison_display.to_string(index=False))


## Figures


## Figures

![Cross-subject performance with participant-bootstrap intervals](../results/full_cohort/figures/performance.png)

![Aggregate confusion matrices](../results/full_cohort/figures/confusion_matrices.png)


## Interpretation and limits

CSP-LDA produced 61.9% mean participant accuracy and band-power logistic regression 58.4% on this selected cohort and fixed protocol. The 3.5 percentage-point paired difference is exploratory. These results do not establish performance on other datasets, clinical use, or diagnostic value. The evaluation uses 104 participants; five database participants were not in the verified local cohort, and no reason is inferred for their absence.

To reproduce the full analysis from the project directory, run:

```bash
python scripts/run_analysis.py --data-root data/raw --out results/full_cohort
```
